<a href="https://colab.research.google.com/github/eceirem/COVID19-Pneumonia-XRay-Classification/blob/main/notebooks/01b_preprocessing_not_masked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
01b_preprocessing_not_masked.py
Author: Ece
Description: Applies Global CLAHE and Unsharp Masking without lung masks for the Ablation Study.
To prevent Google Drive I/O bottlenecks and empty file errors, all processing is done
strictly on Colab's local SSD. It outputs two clean ZIP files ready for download.
"""

import os
import cv2
import glob
import random
import shutil
import zipfile
from tqdm import tqdm

# --- CONFIGURATION & PATHS ---
# Input: Raw Kaggle Zip File located in your Drive
DRIVE_RAW_ZIP = "/content/drive/MyDrive/Sayısal Görüntü İşleme Projesi/Dataset/COVID-19_Radiography_Database.zip"

# Local Colab Working Directories (Ultra-fast I/O)
LOCAL_EXTRACT_DIR = "/content/raw_data"
LOCAL_DL_DIR = "/content/NotMasked/Full_Dataset"
LOCAL_ML_DIR = "/content/NotMasked/Balanced_Dogukan"

# Output ZIP files that will be generated in Colab root
ZIP_DL_OUTPUT = "/content/Full-Data_Preprocessed_Dataset_NotMasked.zip"
ZIP_ML_OUTPUT = "/content/Balanced_Dataset_Dogukan_NotMasked.zip"

TARGET_SIZE = (224, 224) # Standardized for ResNet/ViT/ConvNeXt
CLASSES = ["COVID", "Normal", "Viral Pneumonia"]

def apply_edge_enhancement_and_clahe(img_path, output_path):
    """
    Applies Global CLAHE and Unsharp Masking to highlight parenchymal opacities.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return

    # Standardize resolution
    img = cv2.resize(img, TARGET_SIZE)

    # 1. Global CLAHE (Improves contrast globally)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_img = clahe.apply(img)

    # 2. Unsharp Masking (Edge Enhancement)
    gaussian_blur = cv2.GaussianBlur(clahe_img, (9, 9), 10.0)
    enhanced_img = cv2.addWeighted(clahe_img, 1.5, gaussian_blur, -0.5, 0, clahe_img)

    cv2.imwrite(output_path, enhanced_img)

def zip_directory(folder_path, zip_path):
    """Compresses a directory into a ZIP file."""
    print(f"\n[INFO] Zipping {folder_path} into {zip_path}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Keep directory structure inside the zip
                arcname = os.path.relpath(file_path, os.path.dirname(folder_path))
                zipf.write(file_path, arcname)
    print(f"[SUCCESS] Created {zip_path}")

def process_datasets():
    """
    Extracts raw data, processes images, balances the ML subset, and creates ZIP files.
    """
    print("[INFO] Extracting raw dataset from Drive...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_RAW_ZIP, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)

    # Find the root folder dynamically
    root_raw_dir = LOCAL_EXTRACT_DIR
    for root, dirs, _ in os.walk(LOCAL_EXTRACT_DIR):
        if all(c in dirs for c in CLASSES):
            root_raw_dir = root
            break

    # 1. Process Full Dataset (For Deep Learning)
    print("\n[INFO] Phase 1: Creating Full Dataset (Not Masked)...")
    for cls in CLASSES:
        img_dir = os.path.join(root_raw_dir, cls, "images")
        if not os.path.exists(img_dir): # Fallback
            img_dir = os.path.join(root_raw_dir, cls)

        output_dir = os.path.join(LOCAL_DL_DIR, cls)
        os.makedirs(output_dir, exist_ok=True)

        img_paths = glob.glob(os.path.join(img_dir, "*.png"))
        for img_path in tqdm(img_paths, desc=f"Processing {cls}"):
            filename = os.path.basename(img_path)
            apply_edge_enhancement_and_clahe(img_path, os.path.join(output_dir, filename))

    # 2. Create Balanced Subset (For Machine Learning - Dogukan)
    print("\n[INFO] Phase 2: Creating Balanced Subset...")
    min_samples = min([len(os.listdir(os.path.join(LOCAL_DL_DIR, c))) for c in CLASSES])
    print(f"[INFO] Minority class size is {min_samples}. Undersampling...")

    for cls in CLASSES:
        src_dir = os.path.join(LOCAL_DL_DIR, cls)
        target_dir = os.path.join(LOCAL_ML_DIR, cls)
        os.makedirs(target_dir, exist_ok=True)

        all_images = os.listdir(src_dir)
        selected_images = random.sample(all_images, min_samples)

        for img in selected_images:
            shutil.copy(os.path.join(src_dir, img), os.path.join(target_dir, img))

    # 3. Zip Everything
    print("\n[INFO] Phase 3: Zipping outputs for download...")
    zip_directory(LOCAL_DL_DIR, ZIP_DL_OUTPUT)
    zip_directory(LOCAL_ML_DIR, ZIP_ML_OUTPUT)

if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')
    process_datasets()
    print("\n[DONE] You can now download the two ZIP files directly from the Colab file browser (left panel).")